# Pentax K1000 Light Meter Calibration

This notebook analyzes the variable resistors in the Pentax K1000 light meter system and helps calibrate the light meter circuit for accurate exposure measurement.

## Calibration Process Overview

1. **Variable Resistor Linearity Analysis**: Check aperture, shutter speed, and light meter circuit resistors
2. **Circuit Modeling**: Model the complete circuit and calculate EV (Exposure Value) errors
3. **Resistor Recommendation**: Suggest optimal adjustment resistor values

In [5]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Import calibration utilities
from k1000_calibration import (
    analyze_linearity,
    plot_linearity,
    K1000CircuitModel,
    find_optimal_adjustment_resistor,
    generate_calibration_report
)

## 1. Aperture Variable Resistor Linearity Check

Enter the resistance measurements for different f-stop values. The aperture wheel variable resistor should show linear resistance change across the f-stop range.

In [3]:
# Aperture Resistor Data Entry
# Typical K1000 f-stops: 1.4, 2, 2.8, 4, 5.6, 8, 11, 16, 22
# Add your measured resistance values in the second column

aperture_data = {
    'F-Stop': [1.4, 2.0, 2.8, 4.0, 5.6, 8.0, 11.0, 16.0, 22.0],
    'Resistance (Ω)': [0]*9  # Replace with your measured values
}

aperture_df = pd.DataFrame(aperture_data)
print("Enter aperture resistor measurements:")
print(aperture_df.to_string(index=False))
print("\nUpdate the DataFrame above with your measurements and run the next cell")

Enter aperture resistor measurements:
 F-Stop  Resistance (Ω)
    1.4               0
    2.0               0
    2.8               0
    4.0               0
    5.6               0
    8.0               0
   11.0               0
   16.0               0
   22.0               0

Update the DataFrame above with your measurements and run the next cell


In [ ]:
# Analyze Aperture Resistor Linearity
aperture_analysis = analyze_linearity(
    np.array(aperture_df['F-Stop']), 
    np.array(aperture_df['Resistance (Ω)']), 
    label="Aperture"
)

print("=== APERTURE RESISTOR LINEARITY ANALYSIS ===")
print(f"R² Value: {aperture_analysis['r_squared']:.4f} (1.0 = perfect linearity)")
print(f"RMS Error: {aperture_analysis['rms_error']:.2f} Ω")
print(f"Maximum Deviation: {aperture_analysis['max_deviation']:.2f} Ω")
print(f"Linearity: {aperture_analysis['linearity_percent']:.2f}%")
print(f"Slope: {aperture_analysis['slope']:.4f} Ω/f-stop")

# Only plot if data is available
if not any(np.isnan(np.array(aperture_df['Resistance (Ω)']))):
    plot_linearity(
        np.array(aperture_df['F-Stop']), 
        np.array(aperture_df['Resistance (Ω)']),
        aperture_analysis,
        "Aperture Variable Resistor"
    )
else:
    print("\nNo data plotted - please enter resistance measurements first.")

## 2. Shutter Speed Resistor Linearity Check

Enter resistance measurements for different shutter speeds at 100 ASA. The shutter speed cam should produce linear resistance change across the speed range.

In [ ]:
# Shutter Speed Resistor Data Entry at 100 ASA
# Typical K1000 speeds: 1, 2, 4, 8, 15, 30, 60, 125, 250, 500, 1000

shutter_data = {
    'Shutter Speed (1/s)': [1, 2, 4, 8, 15, 30, 60, 125, 250, 500, 1000],
    'Resistance (Ω)': [np.nan]*11  # Replace with your measured values
}

shutter_df = pd.DataFrame(shutter_data)
print("Enter shutter speed resistor measurements (at 100 ASA):")
print(shutter_df.to_string(index=False))
print("\nUpdate the DataFrame above with your measurements and run the next cell")

In [ ]:
# Analyze Shutter Speed Resistor Linearity
# Use log scale for shutter speeds (they are logarithmic)
shutter_log = np.log2(np.array(shutter_df['Shutter Speed (1/s)']))

shutter_analysis = analyze_linearity(
    shutter_log,
    np.array(shutter_df['Resistance (Ω)'])
)

print("=== SHUTTER SPEED RESISTOR LINEARITY ANALYSIS ===")
print(f"R² Value: {shutter_analysis['r_squared']:.4f} (1.0 = perfect linearity)")
print(f"RMS Error: {shutter_analysis['rms_error']:.2f} Ω")
print(f"Maximum Deviation: {shutter_analysis['max_deviation']:.2f} Ω")
print(f"Linearity: {shutter_analysis['linearity_percent']:.2f}%")
print(f"Slope: {shutter_analysis['slope']:.4f} Ω/LOG₂(speed)")

# Plot if data available
if not any(np.isnan(np.array(shutter_df['Resistance (Ω)']))):
    plot_linearity(
        shutter_log,
        np.array(shutter_df['Resistance (Ω)']),
        shutter_analysis,
        "Shutter Speed Variable Resistor (log scale)"
    )
else:
    print("\nNo data plotted - please enter resistance measurements first.")

## 3. Light Meter Circuit Resistance Analysis

Enter the measured resistance of the light meter circuit at different EV (Exposure Value) levels. Each EV step represents a one-stop change in exposure.

In [ ]:
# Light Meter Circuit Resistance Data Entry
# EV values represent different lighting conditions
# Typical range: EV 2-16 for normal photography

light_meter_data = {
    'EV Level': [2, 4, 6, 8, 10, 12, 14, 16],
    'Circuit Resistance (Ω)': [np.nan]*8  # Replace with your measured values
}

lightem_df = pd.DataFrame(light_meter_data)
print("Enter light meter circuit resistance measurements:")
print(lightem_df.to_string(index=False))
print("\nUpdate the DataFrame above with your measurements and run the next cell")

In [ ]:
# Analyze Light Meter Circuit Linearity
lm_analysis = analyze_linearity(
    np.array(lightem_df['EV Level']),
    np.array(lightem_df['Circuit Resistance (Ω)'])
)

print("=== LIGHT METER CIRCUIT LINEARITY ANALYSIS ===")
print(f"R² Value: {lm_analysis['r_squared']:.4f} (1.0 = perfect linearity)")
print(f"RMS Error: {lm_analysis['rms_error']:.2f} Ω")
print(f"Maximum Deviation: {lm_analysis['max_deviation']:.2f} Ω")
print(f"Linearity: {lm_analysis['linearity_percent']:.2f}%")
print(f"Slope: {lm_analysis['slope']:.4f} Ω/EV")

# Plot if data available
if not any(np.isnan(np.array(lightem_df['Circuit Resistance (Ω)']))):
    plot_linearity(
        np.array(lightem_df['EV Level']),
        np.array(lightem_df['Circuit Resistance (Ω)']),
        lm_analysis,
        "Light Meter Circuit Resistance"
    )
else:
    print("\nNo data plotted - please enter resistance measurements first.")

## 4. Circuit Modeling and EV Error Analysis

This section models the complete light meter circuit and calculates the exposure error (in stops/EV) that results from resistance variations.

In [ ]:
# Calculate and Visualize EV Error
if not any(np.isnan(np.array(lightem_df['Circuit Resistance (Ω)']))):
    # Calculate EV errors from measured vs ideal linear relationship
    measured_r = np.array(lightem_df['Circuit Resistance (Ω)'])
    ev_levels = np.array(lightem_df['EV Level'])
    
    # Expected resistance following perfect linear relationship
    expected_r = lm_analysis['slope'] * ev_levels + lm_analysis['intercept']
    
    # Calculate EV error for each measurement
    ev_errors = []
    for i, (meas, exp) in enumerate(zip(measured_r, expected_r)):
        if exp > 0 and meas > 0:
            ev_error = np.log2(meas / exp)
        else:
            ev_error = 0
        ev_errors.append(ev_error)
    
    ev_errors = np.array(ev_errors)
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    
    # Plot 1: Measured vs Expected Resistance
    ax1.scatter(ev_levels, measured_r, label='Measured', s=80, alpha=0.7, color='blue')
    ax1.plot(ev_levels, expected_r, 'r--', label='Linear Fit', linewidth=2)
    ax1.set_xlabel('EV Level')
    ax1.set_ylabel('Resistance (Ω)')
    ax1.set_title('Light Meter Circuit: Measured vs Expected Resistance')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: EV Error
    colors = ['red' if e > 0.1 else 'orange' if e > 0.05 else 'green' for e in np.abs(ev_errors)]
    ax2.bar(ev_levels, ev_errors, color=colors, alpha=0.7, edgecolor='black')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.axhline(y=0.1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='±0.1 EV threshold')
    ax2.set_xlabel('EV Level')
    ax2.set_ylabel('EV Error (stops)')
    ax2.set_title('Exposure Error Across EV Range')
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Print error statistics
    print("\n=== EV ERROR ANALYSIS ===")
    print(f"Maximum EV Error: {np.max(np.abs(ev_errors)):.3f} stops")
    print(f"RMS EV Error: {np.sqrt(np.mean(ev_errors**2)):.3f} stops")
    print(f"Mean Signed Error: {np.mean(ev_errors):.3f} stops")
    print(f"\nError Distribution:")
    print(f"  Within ±0.05 EV: {np.sum(np.abs(ev_errors) <= 0.05)} points")
    print(f"  Within ±0.1 EV:  {np.sum(np.abs(ev_errors) <= 0.1)} points")
    print(f"  > ±0.1 EV:       {np.sum(np.abs(ev_errors) > 0.1)} points")
else:
    print("No light meter data entered yet. Please fill in measurements to see EV error analysis.")

## 5. Adjustment Resistor Recommendation

This section suggests optimal adjustment resistor values to minimize exposure error across the operating range.

In [ ]:
# Calculate Optimal Adjustment Resistor
if not any(np.isnan(np.array(lightem_df['Circuit Resistance (Ω)']))):
    measured_r = np.array(lightem_df['Circuit Resistance (Ω)'], dtype=float)
    ev_vals = np.array(lightem_df['EV Level'], dtype=float)
    
    optimal_r, max_error, status = find_optimal_adjustment_resistor(measured_r, ev_vals)
    
    print("=== ADJUSTMENT RESISTOR RECOMMENDATION ===")
    if status == "success":
        print(f"Optimal adjustment resistor: {optimal_r:.1f} Ω")
        print(f"Maximum EV error with adjustment: {max_error:.3f} stops")
        print(f"\nConfiguration suggestions:")
        print(f"  - Use {optimal_r:.0f} Ω trimmer potentiometer (closest standard value)")
        print(f"  - Connect in parallel with main circuit for fine adjustment")
        print(f"  - Expected accuracy: ±{max_error*2:.2f} EV (±{max_error*2*100:.1f}% exposure error)")
    else:
        print("Cannot calculate recommendation - insufficient data")
        optimal_r = None
else:
    print("Please enter light meter circuit resistance measurements first.")
    optimal_r = None